# 뉴스 데이터 xml 수집하기 - sbs news

In [ ]:
import requests
from bs4 import BeautifulSoup

news_rss = requests.get('https://news.sbs.co.kr/news/SectionRssFeed.do?sectionId=07')
news_rss_soup = BeautifulSoup(news_rss.content, 'xml')
link_list = news_rss_soup.select('item > link')
print("기사 개수:", len(link_list))

In [ ]:
title_list = news_rss_soup.select('item > title')
title_list = [title.text for title in title_list]
print(title_list)

In [ ]:
news_data = []
for link in link_list:
    
    retry = 0 
    while (True) :
        news_response = requests.get(link.text, timeout=10)
        if news_response.status_code == 200 :
            break
        else :
            retry += 1
            if retry > 3 : break
            
    if news_response.status_code != 200 :
        print("### 기사를 가져오는 데 실패했습니다.!!!")
        print("URL :", link.text)
        continue
    
    print('-'*100)
    news_soup = BeautifulSoup(news_response.content, 'html.parser')
    mews_title = news_soup.select_one('div.w_article_title > #news-title').text   
    print("제목 :", mews_title)     
    news_content = news_soup.select_one("div.text_area[itemprop='articleBody']").text
    print("본문 :\n", news_content[:100])
    
    
    #news-title
    #container > div.w_inner > div.w_article > div.w_article_cont > div.w_article_left > div.article_cont_area > div.main_text > div

# import pandas as pd
# news_df = pd.DataFrame(data={'title': title_list, 'content': news_data})
# print(news_df.head())

# news_df.to_csv("news.csv", encoding="utf-8-sig", index=False)
# print("Save complete")

In [ ]:
# print(news_response.url)
# print(news_response.status_code)
# print(news_content_soup.prettify()[:2000])  # 앞부분만 확인

# 뉴스 컨텐츠 클린징

In [ ]:
news_data_1 = []
for link in link_list:
    news_response = requests.get(link.text)
    news_content_soup = BeautifulSoup(news_response.content, 'html.parser')

    news_content = news_content_soup.select_one("div[itemprop=articleBody]")

    if news_content is not None:
          news_data_1.append(news_content.text.strip())
    else:
        news_data_1.append("본문 없음")  # 또는 None, ""
news_data_1_df = pd.DataFrame(data={'title': title_list, 'content': news_data_1})

In [ ]:
news_data_1_df.head()

In [ ]:
missing_count = sum(1 for content in news_data_1 if content == "본문 없음")
print(f"본문이 없는 기사 개수: {missing_count}개")
print(f"전체 기사 개수: {len(news_data_1)}개")

In [ ]:
import pandas as pd

news_df = pd.DataFrame({
    'title': title_list,
    'content': news_data
})
news_df['is_missing'] = news_df['content'] == "본문 없음"
missing_count = news_df['is_missing'].sum()
total_count = len(news_df)

print(f"본문 없음: {missing_count}개 / 전체: {total_count}개 ({missing_count/total_count:.1%})")